# Phase 3: Route Maker and Graph Processing

**Student:** Tanishq  
Build the customer graph, run Dijkstra, then approximate a delivery tour.

## Theory

Dijkstra computes shortest paths from one source on a graph with non-negative edge weights.

Nearest-neighbor TSP:
1. Start at the warehouse.
2. Go to the closest remaining selected destination (shortest-path distance).
3. Repeat until every selected destination is visited.
4. Return to the warehouse.

The tour is approximate, not guaranteed optimal.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from app import format_graph, load_graph, load_parcels
from src.dijkstra import dijkstra, shortest_path
from src.knapsack import knapsack_select_parcels
from src.sorting import sort_parcels_by_priority_and_deadline
from src.tsp import expand_route_with_shortest_paths, plan_nearest_neighbor_route

graph = load_graph(ROOT / "datasets/routes.csv")
parcels = sort_parcels_by_priority_and_deadline(load_parcels(ROOT / "datasets/parcels.csv"))
_, selected, _ = knapsack_select_parcels(parcels, 12)
destinations = list(dict.fromkeys(parcel.destination for parcel in selected))

print("\n".join(format_graph(graph)))
print()
print("Selected destinations:", ", ".join(destinations))

In [ ]:
distances, previous = dijkstra(graph, "Warehouse")
print("Shortest paths from Warehouse:")
for destination in destinations:
    path = shortest_path(previous, destination)
    print(f"  {' -> '.join(path)} | {distances[destination]} km")

In [ ]:
route, distance = plan_nearest_neighbor_route(graph, "Warehouse", destinations)
expanded = expand_route_with_shortest_paths(graph, route)
print("Nearest-neighbor stop order:", " -> ".join(route))
print("Expanded road path:", " -> ".join(expanded))
print(f"Total route distance: {distance} km")
assert route[0] == route[-1] == "Warehouse"
assert distance == 23
print("Route checks passed.")

## Phase 3 summary

The sample tour is `Warehouse -> B -> A -> D -> F -> I -> Warehouse` at 23 km. Dijkstra supplies each hop; nearest-neighbor only chooses the next stop.